In [1]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
from transformers import ASTForAudioClassification
from transformers import ASTModel
from transformers import DefaultDataCollator
#from datasets import load_metric
import evaluate
from transformers import Trainer, TrainingArguments
import os
from transformers import EarlyStoppingCallback

c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0512 12:51:18.364000 26868 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
data_path = Path(r"C:\Users\Kochana\projects\genres\data\gtzan\gtzan.npz")
data = np.load(data_path)
lst = data.files

In [3]:
tracks_path = []
labels = []
#data_path = Path(r"/content/drive/MyDrive/data/gtzan_old")
for path in lst:
    #path_file = os.path.join(data_path, file, "track.npy")
    tracks_path.append(path)
    labels.append(path[6:][:-12])
le = LabelEncoder()
encoded_labels = le.fit_transform(labels)
train, validation, train_labels, val_labels = train_test_split(
        tracks_path, encoded_labels, test_size=0.2, stratify=encoded_labels, random_state=42)


In [ ]:
class GTZANSpectrogramDataset(Dataset):
    def __init__(self, path, labels):
        self.paths = path
        self.labels = labels
        self.max_time = 1020
        self.data = data
        
    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        #spec = self.spectrograms[idx]  # shape: (128, time)
        spec = data[self.paths[idx]]
        if spec.ndim == 3 and spec.shape[0] == 1:
            spec = spec.squeeze(0)
        spec = spec[:self.max_time, :]
        spec = torch.tensor(spec, dtype=torch.float32)

        #spec = spec.unsqueeze(0)
        label = self.labels[idx]
        return {"input_values": spec, "labels": int(label)}
data_collator = DefaultDataCollator()
#metric = load_metric("accuracy")
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return metric.compute(predictions=predictions, references=labels)
training_args = TrainingArguments(
    output_dir="./ast-gtzan_w_pc",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=3e-5,
    num_train_epochs=80,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    gradient_accumulation_steps=8,
    greater_is_better=True,
    report_to="wandb",
    push_to_hub=True,
    hub_model_id="polinaZaroko/ast-gtzan_w_pc",
    hub_strategy="checkpoint",
    save_total_limit=2,
    warmup_ratio=0.1  #proportion of training to be dedicated to a linear warmup where learning rate gradually increases.
     
)
train_dataset = GTZANSpectrogramDataset(train, train_labels)
val_dataset = GTZANSpectrogramDataset(validation, val_labels)

In [12]:
import wandb
wandb.init(project="ast_model", name="wadims_pc_astmodel_whole_model", config={
            "epochs": training_args.num_train_epochs,
    "batch_size": training_args.per_device_train_batch_size,
    "lr": training_args.learning_rate,
    "model": "AST",
    "augmentation": False,
    "early_stopping" :8
   })

In [13]:
import torch.nn as nn
import torch.nn.functional as F

class ASTForGenreClassification(nn.Module):
    def __init__(self, ast_model, num_labels=10):
        super().__init__()
        self.ast = ast_model
        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(768, num_labels)

    def forward(self, input_values, labels=None):
        #with torch.no_grad():
        x = self.ast.embeddings(input_values)  # patchify
        x = self.ast.encoder(x).last_hidden_state  # (B, T, 768)
        x = x.mean(dim=1)  # simple mean pooling (or use CLS)
        logits = self.classifier(x)

        if labels is not None:
            loss = F.cross_entropy(logits, labels, label_smoothing=0.1)
            return loss, logits
        else:
            return logits


In [14]:
base_ast = ASTModel.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    ignore_mismatched_sizes=True
)
model = ASTForGenreClassification(ast_model=base_ast, num_labels=10)

c:\Users\Kochana\projects\genres\ast_venv\Lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=None,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=8)],
)
trainer.train()

                                                   
  1%|▏         | 50/4000 [00:49<57:17,  1.15it/s]

{'eval_loss': 2.076613664627075, 'eval_accuracy': 0.365, 'eval_runtime': 4.5214, 'eval_samples_per_second': 44.234, 'eval_steps_per_second': 22.117, 'epoch': 1.0}


                                                   
  2%|▎         | 100/4000 [01:38<55:24,  1.17it/s]

{'eval_loss': 1.6059314012527466, 'eval_accuracy': 0.56, 'eval_runtime': 4.4071, 'eval_samples_per_second': 45.381, 'eval_steps_per_second': 22.691, 'epoch': 2.0}


                                                    
  4%|▍         | 150/4000 [02:26<54:46,  1.17it/s]

...

{'eval_loss': 0.8972795009613037, 'eval_accuracy': 0.84, 'eval_runtime': 4.4233, 'eval_samples_per_second': 45.215, 'eval_steps_per_second': 22.608, 'epoch': 19.0}


 25%|██▌       | 1000/4000 [16:09<42:30,  1.18it/s] 

{'loss': 0.5338, 'grad_norm': 0.42736750841140747, 'learning_rate': 2.5008333333333332e-05, 'epoch': 20.0}


                                                   
 25%|██▌       | 1000/4000 [16:14<42:30,  1.18it/s]

{'eval_loss': 0.8486914038658142, 'eval_accuracy': 0.865, 'eval_runtime': 4.4026, 'eval_samples_per_second': 45.427, 'eval_steps_per_second': 22.714, 'epoch': 20.0}


In [10]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_summary(device=0))

True
NVIDIA GeForce RTX 4070 SUPER
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   3825 MiB |   3850 MiB |   5968 MiB |   2142 MiB |
|       from large pool |   3821 MiB |   3847 MiB |   5892 MiB |   2070 MiB |
|       from small pool |      3 MiB |      4 MiB |     76 MiB |     72 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   3825 MiB |   3850 MiB |   5968 MiB |   2142 MiB |
|       from large pool |   3

In [6]:
print(torch.cuda.is_available())

True


In [ ]:
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [17]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


^C
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

^C
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Zugriff verweigert: 'C:\\Users\\Kochana\\projects\\genres\\ast_venv\\Lib\\site-packages\\~orch\\lib\\asmjit.dll'
Check the permissions.


[notice] A new release of pip available: 22.3 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cu126
  Obtaining dependency information for torchvision from https://download.pytorch.org/whl/cu126/torchvision-0.22.0%2Bcu126-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torchaudio from https://download.pytorch.org/whl/cu126/torchaudio-2.7.0%2Bcu126-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torch from https://download.pytorch.org/whl/cu126/torch-2.7.0%2Bcu126-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for pillow!=8.3.*,>=5.3.0 from https://download.pytorch.org/whl/pillow-11.0.0-cp311-cp311-win_amd64.whl.metadata
  Using cached https://download.pytorch.org/whl/pillow-11.0.0-cp311-cp311-win_amd64.whl.metadata (9.3 kB)
   ---------------------------------------- 6.3/6.3 MB 6.6 MB/s eta 0:00:00
   ---------------------------------------- 2.8/2.8 GB 994.8 kB/s eta 0:00:00
   ---------------------------------------- 4.2/4.2 MB 7.1 MB/s eta 0:00:00
  

In [1]:
import torch
print(torch.__version__)

2.7.0+cpu


In [25]:
pip install accelerate==0.28.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip
